# Number of fragment ions per clustered spectrum

**Produces:** `fragment_counts_per_scan_post_falcon.tsv`, the y axis of Supplementary Figure 5.

**Input:** Falcon representative-spectrum MGFs (one per core and hydroxylation state).

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import pandas as pd

In [ ]:
# Path of each MGF file for each bile acid class
mgf_files = {
    "Aceto_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/aceto_dihydroxy_falcon_eps01_minsamples2_scan.mgf",

    "C23_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_dihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C23_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_monohydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C23_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_pentahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C23_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C23_trihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_trihydroxy_falcon_eps01_minsamples2_scan.mgf",

    "C24_alkylamine_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_dihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_monohydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_pentahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_part1": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part1_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_part2": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part2_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_part3": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part3_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_part4": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part4_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_combined": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_trihydroxy_falcon_eps01_minsamples2_combined_renumbered.mgf",

    "C24_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_dihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_monohydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_nonhydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_nonhydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_pentahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_trihydroxy_part1": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part1_falcon_eps01_minsamples2_scan.mgf",
    "C24_trihydroxy_part2": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part2_falcon_eps01_minsamples2_scan.mgf",
    "C24_trihydroxy_part3": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part3_falcon_eps01_minsamples2_scan.mgf",
    "C24_trihydroxy_combined": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_final_trihydroxy_combined_renumbered.mgf",

    "C27_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_dihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C27_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_monohydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C27_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_pentahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C27_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C27_trihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_trihydroxy_falcon_eps01_minsamples2_scan.mgf",

    "C28_alcohol_1OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_1OH_falcon_eps01_minsamples2_scan.mgf",
    "C28_alcohol_2OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_2OH_falcon_eps01_minsamples2_scan.mgf",
    "C28_alcohol_3OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_3OH_falcon_eps01_minsamples2_scan.mgf",
}

In [ ]:
from pathlib import Path
import time

def _first_int_from_bytes(value: bytes):
    """Return first integer found in a byte string, or None if absent."""
    n = len(value)
    i = 0
    while i < n:
        b = value[i]
        if 48 <= b <= 57:  # digit
            j = i + 1
            while j < n and 48 <= value[j] <= 57:
                j += 1
            return int(value[i:j])
        i += 1
    return None

def count_fragments_per_scan(mgf_path):
    rows = []
    in_spectrum = False
    fragment_count = 0
    current_scan = None
    spectrum_index = 0

    # Large buffer helps with very large MGF files.
    with open(mgf_path, "rb", buffering=1024 * 1024) as handle:
        for raw_line in handle:
            line = raw_line.rstrip(b"\r\n")
            if not line:
                continue

            if line == b"BEGIN IONS":
                in_spectrum = True
                fragment_count = 0
                current_scan = None
                spectrum_index += 1
                continue

            if not in_spectrum:
                continue

            if line == b"END IONS":
                rows.append((
                    current_scan if current_scan is not None else spectrum_index,
                    fragment_count,
                ))
                in_spectrum = False
                continue

            if line.startswith(b"SCANS="):
                current_scan = _first_int_from_bytes(line[6:])
                continue

            # Metadata lines contain '='. Peak lines start with m/z value.
            if b"=" not in line:
                first = line[0]
                if 48 <= first <= 57:  # starts with a number
                    fragment_count += 1

    return rows

In [ ]:
all_rows = []
missing_files = []
total_files = len(mgf_files)
t0_all = time.perf_counter()

for i, (ba_class, mgf_path) in enumerate(mgf_files.items(), start=1):
    path_obj = Path(mgf_path)
    if not path_obj.exists():
        print(f"[{i}/{total_files}] Missing file for {ba_class}: {mgf_path}")
        missing_files.append({"BA_class": ba_class, "mgf_file": mgf_path})
        continue

    print(f"[{i}/{total_files}] Processing {ba_class} ...")
    t0 = time.perf_counter()
    rows = count_fragments_per_scan(mgf_path)
    dt = time.perf_counter() - t0

    all_rows.extend((scan, n_fragments, ba_class, mgf_path) for scan, n_fragments in rows)
    print(f"[{i}/{total_files}] Done {ba_class}: {len(rows)} scans in {dt:.1f}s")

fragments_df = pd.DataFrame(
    all_rows,
    columns=["scan_number", "n_fragments", "BA_class", "mgf_file"],
)

print(f"Loaded BA classes: {total_files - len(missing_files)}")
print(f"Missing files: {len(missing_files)}")
if missing_files:
    missing_files_df = pd.DataFrame(missing_files)
    display(missing_files_df)

display(fragments_df.head())
print(f"Total scans: {len(fragments_df)}")
print(f"Total elapsed: {time.perf_counter() - t0_all:.1f}s")

In [ ]:
fragments_df.to_csv(f"{PROJECT_DIR}/Entropy/fragment_counts_per_scan_post_falcon.tsv", sep="\t", index=False)